### Задание: Регрессия. Предсказание количества операций

* подготовка данных
  * проверка на пропуски данных
  * построение графиков
  * разделение выборки 80/20
* обучение модели
  * обучить модель **LinearRegression**
* оценка точности модели
  * рассчитать R², MAE, RMSE
* выводы по работе

In [ ]:
# импорт библиотек
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")

#### Конфигурация

Выбираем **один** resist_id и **одну** операцию (operation_id) для предсказания fact_value (количество операций).

In [ ]:
# --- Выбор resist_id и operation_id (можно менять) ---
RESIST_ID = 124
OPERATION_ID = "8823"  # строка — как в operation_parameter

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_SIZE = 0.2  # 80/20
RANDOM_STATE = 42

#### Загрузка данных, оценка и подготовка

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_division_title(div_json: str):
    if pd.isna(div_json):
        return None
    try:
        obj = json.loads(div_json)
    except Exception:
        return None
    return obj.get("name")


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def get_calc_value(calc_obj: dict, key: str):
    val = calc_obj.get(key)
    if isinstance(val, dict):
        return val.get("value")
    return val


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    """Разворачивает calculate_parameter в плоский словарь: все поля → колонки calc_<key>."""
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


# Загрузка CSV
dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(
        f"Файлы данных не найдены в {RAW_DIR}. "
        "Убедитесь, что папка data/raw существует и содержит data-2022.csv и др."
    )

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "division_title": parse_division_title(row.get("division_name")),
            "id": row.get("id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

# Приведение calc_* колонок к числу (где возможно)
calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print("Развёрнутые поля calculate_parameter:", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

# Фильтр: только выбранные resist_id и operation_id
df = df_model39[
    (df_model39["resist_id"] == RESIST_ID) & (df_model39["operation_id"].astype(str) == str(OPERATION_ID))
].copy()

if df.empty:
    raise ValueError(
        f"Нет данных для resist_id={RESIST_ID}, operation_id={OPERATION_ID}. "
        f"Доступные resist_id: {sorted(df_model39['resist_id'].dropna().unique()[:10].tolist())}..., "
        f"operation_id: {df_model39['operation_id'].unique()[:5].tolist()}..."
    )

# Feature Engineering (как в catboost_per_resist_id)
group = df.groupby("operation_id")
df["lag_1"] = group["fact_value"].shift(1)
df["lag_12"] = group["fact_value"].shift(12)
df["roll_mean_3"] = group["fact_value"].transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
df["roll_mean_6"] = group["fact_value"].transform(lambda s: s.shift(1).rolling(6, min_periods=6).mean())
df["trend"] = (df["lag_1"] / df["roll_mean_3"]).replace([np.inf, -np.inf], 0)

num_cols_clean = df.select_dtypes(include=["number"]).columns
df[num_cols_clean] = df[num_cols_clean].fillna(0)
df = df.infer_objects(copy=False)

df.head()

#### Проверка пропусков данных

In [ ]:
# проверка пропусков значений
if df.isnull().sum().sum() != 0:
    print("Обнаружены пропуски:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    df = df.fillna(0)
else:
    print("dataset не содержит пустых значений")

#### Оценка датасета и построение графиков

In [ ]:
df.describe().T

In [ ]:
# График: fact_value (количество операций) по времени
df_plot = df.copy()
df_plot["date_plot"] = pd.to_datetime(df_plot["date"])
plt.figure(figsize=(10, 4))
plt.plot(df_plot["date_plot"], df_plot["fact_value"], marker="o", markersize=4)
plt.xlabel("Дата")
plt.ylabel("fact_value (количество операций)")
plt.title(f"Динамика fact_value для resist_id={RESIST_ID}, operation_id={OPERATION_ID}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Распределение целевой переменной
sns.histplot(df["fact_value"], bins=15, kde=True)
plt.xlabel("fact_value")
plt.ylabel("Частота")
plt.title("Распределение количества операций")
plt.show()

#### Разделение выборки 80/20

In [ ]:
# Признаки (числовые) для LinearRegression
FEATURE_COLS = [
    "month", "year", "workDayMonth_2", "calculate_minute", "coefficient_fulltime_staff",
    "lag_1", "lag_12", "roll_mean_3", "roll_mean_6", "trend",
]
TARGET_COL = "fact_value"

X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
)

print(f"Обучающая выборка: {len(X_train)} строк")
print(f"Тестовая выборка: {len(X_test)} строк")

#### Обучение модели LinearRegression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

#### Оценка точности предсказания

In [ ]:
y_pred = model.predict(X_test)
y_pred = np.maximum(y_pred, 0)  # предсказания не должны быть отрицательными

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² (коэффициент детерминации): {r2:.4f}")
print(f"MAE (средняя абсолютная ошибка): {mae:.4f}")
print(f"RMSE (среднеквадратичная ошибка): {rmse:.4f}")

# График: факт vs предсказание
plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="Идеальная линия")
plt.xlabel("Фактическое значение")
plt.ylabel("Предсказание")
plt.title("Факт vs предсказание LinearRegression")
plt.legend()
plt.tight_layout()
plt.show()

#### Выводы по работе

* **R²** показывает долю объяснённой дисперсии: чем ближе к 1, тем лучше модель.
* **MAE** — средняя абсолютная ошибка в единицах целевой переменной.
* **RMSE** — штрафует сильнее за крупные ошибки.

Сравнение можно проводить, меняя `RESIST_ID`, `OPERATION_ID` или добавляя другие признаки.